<div style="background:#1e1e2e; padding:16px 20px; border-radius:8px; font-family:monospace; color:#cdd6f4; line-height:1.8"><p style="margin:0 0 8px 0; color:#cba6f7; font-weight:bold; font-size:1.05em;">✅ Session 10 — Typing &amp; Dataclasses · Solutions</p><p style="margin:0;">Worked, runnable solutions for the 12 <strong>Exercises</strong> and 8 <strong>Code Challenges</strong>. Run top to bottom to verify. Try them in <code>01_typing.ipynb</code> first.</p></div>

In [ ]:
from dataclasses import dataclass, field, asdict, replace
from typing import Optional, Callable, TypeVar, TypedDict, Protocol, Literal, Generic
from collections import Counter

### Exercises — Solutions

In [ ]:
# E1 — annotate a function
def greet(name: str) -> str:
    return f"hi {name}"

print(greet("Ada"))                     # hi Ada

In [ ]:
# E2 — annotate a container param
def mean(xs: list[float]) -> float:
    return sum(xs) / len(xs)

print(mean([2.0, 4.0]))                 # 3.0

In [ ]:
# E3 — return int | None
def first_positive(xs: list[int]) -> int | None:
    for x in xs:
        if x > 0:
            return x
    return None

print(first_positive([-1, -2, 5]), first_positive([-1]))   # 5 None

In [ ]:
# E4 — Union parameter
def to_int(x: int | str) -> int:
    return int(x)

print(to_int(3), to_int("7"))           # 3 7

In [ ]:
# E5 — typed Callable parameter
def repeat_apply(f: Callable[[int], int], x: int, n: int) -> int:
    for _ in range(n):
        x = f(x)
    return x

print(repeat_apply(lambda v: v + 1, 0, 5))   # 5

In [ ]:
# E6 — generic with TypeVar
T = TypeVar("T")
def last(xs: list[T]) -> T:
    return xs[-1]

print(last([1, 2, 3]), last(["a", "b"]))     # 3 b

In [ ]:
# E7 — basic dataclass (free __repr__ / __eq__)
@dataclass
class Book:
    title: str
    author: str
    year: int

b = Book("Dune", "Herbert", 1965)
print(b)                                # Book(title='Dune', author='Herbert', year=1965)
print(b == Book("Dune", "Herbert", 1965))   # True

In [ ]:
# E8 — mutable field via default_factory
@dataclass
class Cart:
    items: list[str] = field(default_factory=list)

c1, c2 = Cart(), Cart()
c1.items.append("apple")
print(c1.items, c2.items)               # ['apple'] []  (independent)

In [ ]:
# E9 — frozen dataclass as a counter key
@dataclass(frozen=True)
class GridCell:
    x: int
    y: int

cells = [GridCell(0, 0), GridCell(1, 1), GridCell(0, 0)]
print(Counter(cells)[GridCell(0, 0)])   # 2

In [ ]:
# E10 — sort with order=True
@dataclass(order=True)
class Student:
    gpa: float
    name: str

ranked = sorted([Student(3.2, "A"), Student(3.9, "B"), Student(3.5, "C")], reverse=True)
print([s.name for s in ranked])         # ['B', 'C', 'A']

In [ ]:
# E11 — TypedDict payload + reader
class Payload(TypedDict):
    user_id: int
    active: bool

def is_active(p: Payload) -> bool:
    return p["active"]

print(is_active({"user_id": 1, "active": True}))   # True

In [ ]:
# E12 — structural typing with Protocol
class HasName(Protocol):
    name: str

def describe(obj: HasName) -> str:
    return f"name={obj.name}"

@dataclass
class Dog:
    name: str

print(describe(Dog("Rex")))             # name=Rex

### Code Challenges — Solutions

In [ ]:
# C1 — frozen + hashable de-dup
@dataclass(frozen=True)
class Money:
    amount: int
    currency: str

coins = {Money(10, "USD"), Money(10, "USD"), Money(5, "EUR")}
print(len(coins))                       # 2  (duplicate collapsed)

In [ ]:
# C2 — type alias + function
JSONNum = list[float]
def normalize(v: JSONNum) -> JSONNum:
    s = sum(v)
    return [x / s for x in v]

print(normalize([1.0, 3.0]))            # [0.25, 0.75]

In [ ]:
# C3 — sort ignoring a field via compare=False
@dataclass(order=True)
class Task:
    priority: int
    label: str = field(compare=False)

print(sorted([Task(2, "b"), Task(1, "a")])[0].label)   # a
print(Task(1, "x") == Task(1, "y"))     # True  (label ignored)

In [ ]:
# C4 — validate in __post_init__
@dataclass
class Percentage:
    value: float
    def __post_init__(self):
        if not 0 <= self.value <= 100:
            raise ValueError("0..100")

print(Percentage(50).value)             # 50
try: Percentage(150)
except ValueError as e: print("guard:", e)   # guard: 0..100

In [ ]:
# C5 — generic Stack[T]
class Stack(Generic[T]):
    def __init__(self) -> None:
        self._items: list[T] = []
    def push(self, x: T) -> None:
        self._items.append(x)
    def pop(self) -> T:
        return self._items.pop()

s: Stack[int] = Stack()
s.push(1); s.push(2)
print(s.pop(), s.pop())                 # 2 1

In [ ]:
# C6 — Protocol over unrelated model classes
class SupportsPredict(Protocol):
    def predict(self, x: float) -> float: ...

class Const:
    def __init__(self, c): self.c = c
    def predict(self, x): return self.c
class Double:
    def predict(self, x): return x * 2

def score(models: list[SupportsPredict], x: float) -> list[float]:
    return [m.predict(x) for m in models]

print(score([Const(5), Double()], 3))   # [5, 6]

In [ ]:
# C7 — immutable update with replace
@dataclass(frozen=True)
class HyperParams:
    lr: float = 0.01
    epochs: int = 10

def with_lr(cfg: HyperParams, lr: float) -> HyperParams:
    return replace(cfg, lr=lr)

base = HyperParams()
tuned = with_lr(base, 0.1)
print(base.lr, tuned.lr, base is tuned) # 0.01 0.1 False

In [ ]:
# C8 — hand-rolled schema validation (Pydantic-lite)
class UserT(TypedDict):
    name: str
    age: int

def validate(d: dict, schema: type) -> bool:
    hints = schema.__annotations__
    return all(k in d and isinstance(d[k], t) for k, t in hints.items())

print(validate({"name": "Ada", "age": 30}, UserT))   # True
print(validate({"name": "Ada"}, UserT))              # False  (missing age)